# 14_calc_EFPs — corrected workflow for FLUXNET Shuttle HH CSV files

Calculates **per-site, per-year** ecosystem functional properties (EFPs) from
`FLUXNET_FLUXMET_HH` files inside FLUXNET Shuttle zip archives.

### Fixes vs previous version
1. Added missing `TimeZoneCalculatorFLUXNET()` definition.
2. Added missing `calc_yearly_efps_full()` — the main per-year orchestrator.
3. Added missing `calc_monthly_meteo()`.
4. Fixed `EFPcalc()` to work **per year** (separate `EFPcalc_year()`), not full-site.
5. Fixed REddyProc API: removed deprecated `.s` / `.n` argument suffixes.
6. Fixed `bigleaf::filter.data()` QC flag handling (FLUXNET 0–3 scale).
7. Fixed test cell: now uses the correct function signature with all arguments.
8. Fixed cell 9 QC checks to match actual output column names.
9. Removed stale references to `flux_all` / `tower_meta` objects from old script.

## 0. Packages and paths

In [1]:
# ============================================================
# 0. Packages and paths
# ============================================================

packages <- c(
  "dplyr", "tidyr", "purrr", "readr", "stringr",
  "lubridate", "tibble", "glue"
)

missing_pkgs <- packages[!packages %in% rownames(installed.packages())]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)
invisible(lapply(packages, library, character.only = TRUE))

# Optional packages used by EFP functions
optional_packages <- c("quantreg", "bigleaf", "REddyProc", "lutz", "sf", "broom", "plyr")
for (p in optional_packages) {
  if (!requireNamespace(p, quietly = TRUE))
    message("Optional package not installed: ", p, ". Related EFPs may return NA.")
}

# ── EDIT THESE PATHS ────────────────────────────────────────
base_dir   <- "/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02"
# Path to a CSV with at least: SITE_ID, LOCATION_LAT, LOCATION_LONG, LOCATION_ELEV
meta_path  <- file.path(base_dir, "fluxnet_site_metadata_clean_with_elevation.csv")

start_year <- 2017
end_year   <- 2025
# ────────────────────────────────────────────────────────────

out_dir   <- file.path(base_dir, "EFP_outputs_corrected")
efp_dir   <- file.path(out_dir, "yearly_EFP_per_site")
meteo_dir <- file.path(out_dir, "monthly_meteo_per_site")
qc_dir    <- file.path(out_dir, "logs")

for (d in c(out_dir, efp_dir, meteo_dir, qc_dir))
  dir.create(d, showWarnings = FALSE, recursive = TRUE)

cat("Base directory:", base_dir, "\n")
cat("Output directory:", out_dir, "\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Base directory: /mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02 
Output directory: /mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/EFP_outputs_corrected 


## 1. Index FLUXMET HH files inside zip archives

In [2]:
# ============================================================
# 1. Build zip index — FLUXNET_FLUXMET_HH files only
# ============================================================

all_zip_files <- list.files(base_dir, pattern = "\\.zip$",
                            recursive = TRUE, full.names = TRUE)

zip_index_list <- list()
bad_zips       <- character()

for (z in all_zip_files) {
  files_inside <- tryCatch(
    unzip(z, list = TRUE)$Name,
    error = function(e) { bad_zips <<- c(bad_zips, z); NULL }
  )
  if (is.null(files_inside)) next

  hh_file <- files_inside[
    stringr::str_detect(files_inside, "FLUXNET_FLUXMET_HH") &
      stringr::str_detect(files_inside, "\\.csv$")
  ]

  if (length(hh_file) > 0)
    zip_index_list[[length(zip_index_list) + 1]] <-
      tibble(zip_path = z, hh_file = hh_file)
}

zip_index <- bind_rows(zip_index_list) %>%
  mutate(site_id = str_extract(hh_file, "[A-Z]{2}-[A-Za-z0-9]{3}")) %>%
  filter(!is.na(site_id)) %>%
  distinct(site_id, zip_path, hh_file, .keep_all = TRUE) %>%
  arrange(site_id)

write_csv(zip_index,        file.path(qc_dir, "fluxmet_hh_zip_index.csv"))
write_csv(tibble(zip_path = bad_zips), file.path(qc_dir, "bad_or_corrupted_zip_files.csv"))

cat("Total zip files:", length(all_zip_files), "\n")
cat("Good sites with FLUXMET HH:", n_distinct(zip_index$site_id), "\n")
cat("Bad/corrupted zip files:", length(bad_zips), "\n")
head(zip_index)

Total zip files: 357 
Good sites with FLUXMET HH: 329 
Bad/corrupted zip files: 24 


zip_path,hh_file,site_id
<chr>,<chr>,<chr>
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/AMF_AR-TF1_FLUXNET_2016-2018_v1.3_r1.zip,AMF_AR-TF1_FLUXNET_FLUXMET_HH_2016-2018_v1.3_r1.csv,AR-TF1
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/AMF_AR-TF2_FLUXNET_2016-2018_v1.3_r1.zip,AMF_AR-TF2_FLUXNET_FLUXMET_HH_2016-2018_v1.3_r1.csv,AR-TF2
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/EUF_AT-Mmg_FLUXNET_2021-2024_v1.3_r1.zip,EUF_AT-Mmg_FLUXNET_FLUXMET_HH_2021-2024_v1.3_r1.csv,AT-Mmg
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/EUF_AT-Nsd_FLUXNET_2018-2022_v1.3_r1.zip,EUF_AT-Nsd_FLUXNET_FLUXMET_HH_2018-2022_v1.3_r1.csv,AT-Nsd
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/EUF_AT-PsM_FLUXNET_2015-2019_v1.3_r1.zip,EUF_AT-PsM_FLUXNET_FLUXMET_HH_2015-2019_v1.3_r1.csv,AT-PsM
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/EUF_AT-Zoe_FLUXNET_2014-2024_v1.3_r1.zip,EUF_AT-Zoe_FLUXNET_FLUXMET_HH_2014-2024_v1.3_r1.csv,AT-Zoe


## 2. Load site metadata

In [3]:
# ============================================================
# 2. Load site metadata
# Required columns: SITE_ID, LOCATION_LAT, LOCATION_LONG, LOCATION_ELEV
# ============================================================

tower_meta <- readr::read_csv(meta_path, show_col_types = FALSE)

# Verify required columns exist
required_meta_cols <- c("SITE_ID", "LOCATION_LAT", "LOCATION_LONG", "LOCATION_ELEV")
missing_cols <- setdiff(required_meta_cols, names(tower_meta))
if (length(missing_cols) > 0)
  stop("Metadata is missing columns: ", paste(missing_cols, collapse = ", "))

cat("Sites in metadata:", nrow(tower_meta), "\n")
cat("Sites in zip index:", n_distinct(zip_index$site_id), "\n")
cat("Sites with both zip and metadata:",
    sum(zip_index$site_id %in% tower_meta$SITE_ID), "\n")

Sites in metadata: 357 
Sites in zip index: 329 
Sites with both zip and metadata: 329 


## 3. Utility functions

Includes the previously missing `TimeZoneCalculatorFLUXNET()`.

In [4]:
# ============================================================
# 3. Utility functions
# ============================================================

# FIX 1: TimeZoneCalculatorFLUXNET was called inside EFPcalc but never defined.
TimeZoneCalculatorFLUXNET <- function(lat, lon) {
  if (!requireNamespace("lutz", quietly = TRUE) ||
      !requireNamespace("sf",   quietly = TRUE)) {
    message("lutz/sf not available — defaulting to UTC (TimeZone = 0)")
    return(0)
  }
  tz_name <- lutz::tz_lookup_coords(lat, lon, method = "accurate", warn = FALSE)
  offset  <- lutz::tz_offset(
    as.POSIXct("2018-07-01 12:00:00", tz = "UTC"), tz_name
  )
  as.numeric(offset$utc_offset_h)
}

# Safe quantile and mean helpers
q_safe    <- function(x, prob) {
  x <- x[is.finite(x)]
  if (length(x) == 0) return(NA_real_)
  as.numeric(stats::quantile(x, probs = prob, na.rm = TRUE, names = FALSE))
}
mean_safe <- function(x) { x <- x[is.finite(x)]; if (!length(x)) NA_real_ else mean(x) }

# Convert -9999 sentinel values to NA; return NA column if column absent
clean_col <- function(df, col) {
  if (col %in% names(df)) dplyr::na_if(df[[col]], -9999) else NA_real_
}

cat("Utility functions defined.\n")

Utility functions defined.


## 4. Read and standardise one FLUXMET HH file

In [5]:
# ============================================================
# 4. read_fluxmet_hh
# DATETIME is based on TIMESTAMP_END (REddyProc convention).
# YEAR / DOY are derived from TIMESTAMP_START so the final
# half-hour of a year stays in the correct year.
# ============================================================

read_fluxmet_hh <- function(zip_path, hh_file, site_id,
                            start_year = 2017, end_year = 2025) {

  dat <- readr::read_csv(unz(zip_path, hh_file),
                         show_col_types = FALSE, progress = FALSE)

  dat_clean <- dat %>%
    mutate(
      SITE_ID  = site_id,
      siteID   = site_id,

      DATETIME_START = lubridate::ymd_hm(as.character(TIMESTAMP_START), tz = "UTC"),
      DATETIME_END   = lubridate::ymd_hm(as.character(TIMESTAMP_END),   tz = "UTC"),
      DATETIME = DATETIME_END,
      DateTime = DATETIME_END,

      YEAR         = lubridate::year(DATETIME_START),
      year         = YEAR,
      MONTH        = lubridate::month(DATETIME_START),
      DOY          = lubridate::yday(DATETIME_START),
      doy          = DOY,
      HOUR_decimal = lubridate::hour(DATETIME_START) +
                     lubridate::minute(DATETIME_START) / 60,

      # Fluxes
      GPP    = clean_col(cur_data(), "GPP_NT_VUT_REF"),
      GPP_NT = GPP,
      RECO   = clean_col(cur_data(), "RECO_NT_VUT_REF"),
      RECO_NT = RECO,
      NEE    = clean_col(cur_data(), "NEE_VUT_REF"),
      NEP    = -NEE,
      LE     = clean_col(cur_data(), "LE_F_MDS"),
      H      = clean_col(cur_data(), "H_F_MDS"),

      # Meteorology
      TA     = clean_col(cur_data(), "TA_F"),
      VPD    = clean_col(cur_data(), "VPD_F"),
      VPD_kPa = VPD / 10,
      SW_IN  = clean_col(cur_data(), "SW_IN_F"),
      PPFD_IN_FROM_SWIN = SW_IN * 2.11,
      P      = clean_col(cur_data(), "P_F"),
      Precip = P,
      PA     = clean_col(cur_data(), "PA_F"),
      WS     = clean_col(cur_data(), "WS_F"),
      USTAR  = clean_col(cur_data(), "USTAR"),
      # NETRAD not always present; fallback handled downstream
      NETRAD = if ("NETRAD" %in% names(cur_data()))
                 dplyr::na_if(NETRAD, -9999) else NA_real_,
      G      = if ("G_F_MDS" %in% names(cur_data()))
                 dplyr::na_if(G_F_MDS, -9999) else NA_real_,
      CO2    = if ("CO2_F_MDS" %in% names(cur_data()))
                 dplyr::na_if(CO2_F_MDS, -9999) else NA_real_,

      # FIX 7: QC columns — map FLUXNET names; use 0 where absent
      # FLUXNET QC scale: 0 = measured, 1 = good gap-fill, 2-3 = poor gap-fill
      NEE_QC   = if ("NEE_VUT_REF_QC" %in% names(cur_data()))
                   dplyr::na_if(NEE_VUT_REF_QC, -9999) else 0L,
      LE_QC    = if ("LE_F_MDS_QC"    %in% names(cur_data()))
                   dplyr::na_if(LE_F_MDS_QC,    -9999) else 0L,
      H_QC     = if ("H_F_MDS_QC"     %in% names(cur_data()))
                   dplyr::na_if(H_F_MDS_QC,     -9999) else 0L,
      TA_QC    = if ("TA_F_QC"         %in% names(cur_data()))
                   dplyr::na_if(TA_F_QC,         -9999) else 0L,
      VPD_QC   = if ("VPD_F_QC"        %in% names(cur_data()))
                   dplyr::na_if(VPD_F_QC,        -9999) else 0L,
      SW_IN_QC = if ("SW_IN_F_QC"      %in% names(cur_data()))
                   dplyr::na_if(SW_IN_F_QC,      -9999) else 0L
    ) %>%
    filter(YEAR >= start_year, YEAR <= end_year) %>%
    group_by(SITE_ID, YEAR) %>%
    arrange(DATETIME_START, .by_group = TRUE) %>%
    mutate(FiveDaySeq = ceiling(row_number() / (48 * 5))) %>%
    ungroup()

  dat_clean
}

cat("read_fluxmet_hh() defined.\n")

read_fluxmet_hh() defined.


## 5. Per-year EFP calculation function

FIX 4 & 6: Replaced the full-site `EFPcalc()` with a proper per-year version.
FIX 5: REddyProc arguments use the current API (no `.s`/`.n` suffixes).

In [6]:
# ============================================================
# 5. EFPcalc_year — calculate EFPs for ONE site-year
# ============================================================
# Arguments:
#   dat_year  : data.frame, one year of half-hourly FLUXMET data
#               (already standardised by read_fluxmet_hh)
#   site      : character site ID
#   yr        : integer year
#   lat, lon  : numeric coordinates
#   elevation : numeric, metres
#   timezone  : numeric UTC offset hours (from TimeZoneCalculatorFLUXNET)
# Returns a one-row data.frame of EFPs.
# ============================================================

EFPcalc_year <- function(dat_year, site, yr, lat, lon, elevation, timezone) {

  GSfilt <- 0.3

  # ── Derived columns needed by bigleaf / REddyProc ────────
  dat_year <- dat_year %>%
    mutate(
      ET = if (requireNamespace("bigleaf", quietly = TRUE))
             bigleaf::LE.to.ET(LE, TA) * 1800 else NA_real_,
      NETRAD = if_else(is.na(NETRAD), SW_IN, NETRAD)  # fallback if absent
    )

  # ── Empty output returned on failure ─────────────────────
  empty_out <- tibble(
    SITE_ID = site, YEAR = yr,
    uWUE = NA_real_, WUE = NA_real_, ETmax = NA_real_,
    precipAvail = NA_character_, Gavail = NA_character_,
    GSmax = NA_real_, CO2avail = NA_character_, G1 = NA_real_,
    EF = NA_real_, EFampl = NA_real_,
    GPPsat = NA_real_, NEPmax = NA_real_,
    Rb = NA_real_, Rbmax = NA_real_, aCUE = NA_real_,
    TZ = timezone, nyears = 1L, status = "failed"
  )

  # ── Require bigleaf ───────────────────────────────────────
  if (!requireNamespace("bigleaf", quietly = TRUE)) {
    warning("bigleaf not installed — returning NA for site-year ", site, "-", yr)
    return(empty_out)
  }

  # ── Growing-season + precip filters ──────────────────────
  # FIX 7: bigleaf::filter.data uses QC columns with quality.ext appended
  # to vars.qc names. Here we pass good.quality = 1 which keeps QC in {0, 1}
  # (measured + good gap-fill on the FLUXNET 0–3 scale).
  # We exclude 'VPD' from vars.qc because VPD_QC can be absent in some files.
  qc_vars_available <- intersect(
    c("TA", "H", "LE", "NEE"),
    gsub("_QC", "", grep("_QC$", names(dat_year), value = TRUE))
  )

  precipAvail <- if (!all(is.na(dat_year$P)) && sum(dat_year$P > 0, na.rm = TRUE) > 0)
    "yes" else "no"

  dat_filtered <- tryCatch(
    bigleaf::filter.data(
      data.frame(dat_year),
      quality.control   = length(qc_vars_available) > 0,
      filter.growseas   = TRUE,
      filter.precip     = precipAvail == "yes",
      GPP = "GPP_NT", doy = "doy", year = "year", tGPP = GSfilt,
      precip = "P", tprecip = 0.1, precip.hours = 24,
      records.per.hour  = 2,
      vars.qc           = qc_vars_available,
      quality.ext       = "_QC",
      good.quality      = 1
    ),
    error = function(e) {
      message("  filter.data error for ", site, "-", yr, ": ", e$message)
      data.frame(dat_year)[0, ]  # empty frame → downstream checks return NA
    }
  )

  # Daytime + u* subset
  aaa <- subset(dat_filtered, SW_IN > 200 & USTAR > 0.2)

  out <- as.list(empty_out)
  out$precipAvail <- precipAvail
  out$TZ          <- timezone

  if (nrow(aaa) < 40) {
    warning("  Too few daytime records (", nrow(aaa), ") for ", site, "-", yr)
    return(as.data.frame(out))
  }

  # ── WUE / uWUE / ETmax ───────────────────────────────────
  wue <- tryCatch({
    bigleaf::WUE.metrics(
      aaa, GPP = "GPP_NT", NEE = "NEE", LE = "LE",
      VPD = "VPD_kPa", Tair = "TA",
      constants = bigleaf::bigleaf.constants()
    )
  }, error = function(e) NULL)

  if (!is.null(wue)) {
    out$uWUE <- wue[["uWUE"]]
    out$WUE  <- wue[["WUE"]]
  }
  out$ETmax <- q_safe(aaa$ET, 0.95)

  # ── Gsmax and G1 ─────────────────────────────────────────
  aaa_gs <- aaa %>% tidyr::drop_na(WS)

  if (nrow(aaa_gs) >= 10) {
    aaa_gs$pressure <- bigleaf::pressure.from.elevation(
      elevation, aaa_gs$TA, aaa_gs$VPD_kPa
    )

    Ga_vec <- tryCatch(
      bigleaf::aerodynamic.conductance(
        aaa_gs, Tair = "TA", pressure = "pressure",
        wind = "WS", ustar = "USTAR", H = "H",
        Rb_model = "Thom_1972"
      )[["Ga_h"]],
      error = function(e) NULL
    )

    if (!is.null(Ga_vec)) {
      Gs <- tryCatch(
        bigleaf::surface.conductance(
          aaa_gs, Ga = Ga_vec, Tair = "TA", pressure = "pressure",
          Rn = "NETRAD",
          G = if (all(is.na(aaa_gs$G))) NULL else "G",
          S = NULL, VPD = "VPD_kPa", LE = "LE",
          missing.G.as.NA = FALSE, missing.S.as.NA = FALSE
        ),
        error = function(e) NULL
      )

      out$Gavail <- if (all(is.na(aaa_gs$G))) "no" else "yes"

      if (!is.null(Gs)) {
        out$GSmax <- q_safe(Gs$Gs_ms, 0.90)

        aaa_gs$Gs_mol <- Gs$Gs_mol
        aaa_gs <- aaa_gs %>%
          tidyr::drop_na(Gs_mol) %>%
          filter(VPD_kPa > 0)

        if (nrow(aaa_gs) >= 40) {
          if (all(is.na(aaa_gs$CO2))) {
            aaa_gs$CO2 <- 400
            out$CO2avail <- "no"
          } else {
            out$CO2avail <- "yes"
          }

          mod_USO <- tryCatch(
            bigleaf::stomatal.slope(
              aaa_gs, model = "USO", GPP = "GPP_NT", Gs = "Gs_mol",
              Tair = "TA", pressure = "pressure", VPD = "VPD_kPa", Ca = "CO2",
              robust.nls = TRUE, nmin = 40, fitg0 = FALSE
            ),
            error = function(e) NULL
          )
          if (!is.null(mod_USO) && requireNamespace("broom", quietly = TRUE))
            out$G1 <- broom::tidy(mod_USO)$estimate[1]
        }
      }
    }
  }

  # ── EF / EFampl ──────────────────────────────────────────
  aaa$EF <- aaa$LE / (aaa$LE + aaa$H)
  aaa_ef  <- aaa[is.finite(aaa$EF), ]
  if (nrow(aaa_ef) > 0) {
    out$EF     <- median(aaa_ef$EF, na.rm = TRUE)
    out$EFampl <- q_safe(aaa_ef$EF, 0.75) - q_safe(aaa_ef$EF, 0.25)
  }

  # ── GPPsat and NEPmax via 5-day light-response curve ─────
  myLRC <- function(datafilt) {
    if (nrow(datafilt) < 10) return(NA_real_)
    if (mean(is.na(datafilt$NEE)) >= 0.8) return(NA_real_)
    tryCatch({
      fitLRC <- bigleaf::light.response(
        datafilt, NEE = "NEE", Reco = "Reco", PPFD = "PPFD", PPFD_ref = 2000
      )
      if (requireNamespace("broom", quietly = TRUE))
        broom::tidy(fitLRC)$estimate[2]
      else fitLRC$coefficients[2]
    }, error = function(e) NA_real_)
  }

  # For GPPsat, use only growing-season filter (no precip filter)
  dat_lrc <- tryCatch(
    bigleaf::filter.data(
      data.frame(dat_year),
      quality.control = length(qc_vars_available) > 0,
      filter.growseas = TRUE, filter.precip = FALSE,
      GPP = "GPP_NT", doy = "doy", year = "year", tGPP = GSfilt,
      precip = "P", tprecip = 0.1, precip.hours = 24,
      records.per.hour = 2,
      vars.qc = qc_vars_available, quality.ext = "_QC", good.quality = 1
    ),
    error = function(e) data.frame(dat_year)[0, ]
  )

  if (nrow(dat_lrc) > 48 * 5) {
    dat_lrc$FiveDaySeq <- ceiling(seq_len(nrow(dat_lrc)) / (48 * 5))
    zzz <- data.frame(
      NEE = dat_lrc$NEE, PPFD = dat_lrc$PPFD_IN_FROM_SWIN,
      Reco = dat_lrc$RECO_NT, FiveDaySeq = dat_lrc$FiveDaySeq
    )
    outGPPsat <- unlist(by(zzz, zzz$FiveDaySeq, myLRC), use.names = FALSE)
    out$GPPsat <- q_safe(outGPPsat, 0.90)
    daytime_nep <- subset(zzz, PPFD > 200 * 2.11)$NEE * -1
    out$NEPmax  <- q_safe(daytime_nep, 0.99)
  }

  # ── Basal respiration (Rb, Rbmax) via REddyProc ──────────
  # FIX 5: Use current REddyProc API (no .s/.n suffixes on arguments)
  if (requireNamespace("REddyProc", quietly = TRUE)) {
    tryCatch({
      ttt <- data.frame(dat_year) %>%
        dplyr::select(
          DateTime, NEE, NEE_QC, SW_IN, SW_IN_QC, TA, TA_QC
        ) %>%
        rename(
          NEE_QC_OK   = NEE_QC,
          SW_IN_QC_OK = SW_IN_QC,
          TA_QC_OK    = TA_QC
        ) %>%
        mutate(
          # Convert FLUXNET QC (0=measured, 1=good fill) to binary 0/1
          NEE_QC_OK   = as.integer(NEE_QC_OK   <= 1),
          SW_IN_QC_OK = as.integer(SW_IN_QC_OK <= 1),
          TA_QC_OK    = as.integer(TA_QC_OK    <= 1),
          # REddyProc expects DateTime at the end of the half-hour
          DateTime = as.POSIXct(DateTime, tz = "UTC")
        )

      ep <- REddyProc::sEddyProc$new(
        site, ttt,
        c("NEE", "NEE_QC_OK", "SW_IN", "SW_IN_QC_OK", "TA", "TA_QC_OK"),
        LatDeg = as.numeric(lat), LongDeg = as.numeric(lon),
        TimeZoneHour = timezone
      )

      # FIX 5: Current REddyProc API — no .s/.n argument suffixes
      ep$sMRFluxPartition(
        FluxVar    = "NEE",
        QFFluxVar  = "NEE_QC_OK",
        QFFluxValue = 0,          # use only QC_OK == 0 (measured)
        TempVar    = "TA",
        QFTempVar  = "TA_QC_OK",
        QFTempValue = 0,
        RadVar     = "SW_IN",
        TRef       = 273.15 + 15,
        suffix     = ""
      )

      Rref <- ep$sTEMP$R_ref
      out$Rb    <- mean_safe(Rref)
      out$Rbmax <- q_safe(Rref, 0.95)

      # ── aCUE ─────────────────────────────────────────────
      dat_year$Rb_hh <- Rref
      dat_cue <- dat_year %>%
        filter(doy %in% dat_lrc$doy) %>%   # growing-season rows only
        group_by(year, doy) %>%
        summarise(
          mean_GPP = mean(GPP_NT, na.rm = TRUE),
          mean_Rb  = mean(Rb_hh,  na.rm = TRUE),
          .groups  = "drop"
        )
      out$aCUE <- median(1 - dat_cue$mean_Rb / dat_cue$mean_GPP, na.rm = TRUE)

    }, error = function(e)
      message("  REddyProc error for ", site, "-", yr, ": ", e$message)
    )
  }

  out$SITE_ID <- site
  out$YEAR    <- yr
  out$nyears  <- 1L
  out$status  <- "ok"

  as.data.frame(out)
}

cat("EFPcalc_year() defined.\n")

EFPcalc_year() defined.


## 6. Monthly meteorology function

FIX 2: `calc_monthly_meteo()` was called in the batch loop but never defined.

In [ ]:
# ============================================================
# 6. calc_monthly_meteo
# ============================================================

calc_monthly_meteo <- function(dat_full) {
  dat_full %>%
    group_by(SITE_ID, YEAR, MONTH) %>%
    summarise(
      TA_mean   = mean_safe(TA),
      TA_p05    = q_safe(TA,    0.05),
      TA_p95    = q_safe(TA,    0.95),
      VPD_mean  = mean_safe(VPD),
      VPD_p05   = q_safe(VPD,  0.05),
      VPD_p95   = q_safe(VPD,  0.95),
      SW_IN_mean = mean_safe(SW_IN),
      SW_IN_p05  = q_safe(SW_IN, 0.05),
      SW_IN_p95  = q_safe(SW_IN, 0.95),
      P_sum     = sum_safe(P),
      P_mean    = mean_safe(P),
      P_p05     = q_safe(P, 0.05),
      P_p95     = q_safe(P, 0.95),
      n_obs     = n(),
      .groups   = "drop"
    )
}

sum_safe <- function(x) { x <- x[is.finite(x)]; if (!length(x)) NA_real_ else sum(x) }

cat("calc_monthly_meteo() defined.\n")

## 7. Yearly EFP orchestrator

FIX 1: `calc_yearly_efps_full()` was called in the batch loop but never defined.
This function loops over years and calls `EFPcalc_year()` for each.

In [8]:
# ============================================================
# 7. calc_yearly_efps_full — orchestrates per-year EFP calculation
# ============================================================

calc_yearly_efps_full <- function(dat_full, site_id, tower_meta_df) {

  # Get site metadata
  smeta <- tower_meta_df[tower_meta_df$SITE_ID == site_id, ][1, ]
  if (nrow(smeta) == 0 || is.na(smeta$LOCATION_LAT)) {
    warning("No metadata for ", site_id, " — skipping")
    return(tibble())
  }

  lat       <- as.numeric(smeta$LOCATION_LAT)
  lon       <- as.numeric(smeta$LOCATION_LONG)
  elevation <- as.numeric(smeta$LOCATION_ELEV)
  timezone  <- TimeZoneCalculatorFLUXNET(lat, lon)

  years <- sort(unique(dat_full$YEAR))
  cat("  Site:", site_id, "| Years:", paste(years, collapse = ", "), "\n")

  results <- purrr::map_dfr(years, function(yr) {
    dat_yr <- dat_full %>% filter(YEAR == yr)

    # Skip year if too few half-hours (< 3 months of data)
    if (nrow(dat_yr) < 48 * 30 * 3) {
      message("  Skipping ", site_id, "-", yr,
              " (only ", nrow(dat_yr), " half-hours)")
      return(tibble(
        SITE_ID = site_id, YEAR = yr, status = "insufficient_data"
      ))
    }

    tryCatch(
      EFPcalc_year(dat_yr, site_id, yr, lat, lon, elevation, timezone),
      error = function(e) {
        message("  ERROR ", site_id, "-", yr, ": ", e$message)
        tibble(SITE_ID = site_id, YEAR = yr, status = "error")
      }
    )
  })

  results
}

cat("calc_yearly_efps_full() defined.\n")

calc_yearly_efps_full() defined.


## 8. Test on a single site-year

FIX 4 & 9: Fixed the call signature and removed references to undefined objects.

In [9]:
# ============================================================
# 8. Quick test on first available site + first year
# ============================================================

# Pick the first site that also has metadata
test_candidates <- zip_index$site_id[zip_index$site_id %in% tower_meta$SITE_ID]
if (length(test_candidates) == 0) stop("No sites found in both zip_index and tower_meta.")
test_site_id <- test_candidates[1]

cat("Test site:", test_site_id, "\n")

# Read the data
test_row <- zip_index[zip_index$site_id == test_site_id, ][1, ]
test_dat <- read_fluxmet_hh(
  zip_path   = test_row$zip_path,
  hh_file    = test_row$hh_file,
  site_id    = test_site_id,
  start_year = start_year,
  end_year   = end_year
)
cat("Rows loaded:", nrow(test_dat), "| Years:",
    paste(sort(unique(test_dat$YEAR)), collapse = ", "), "\n")

# Get metadata for this site
test_meta <- tower_meta[tower_meta$SITE_ID == test_site_id, ][1, ]
test_lat  <- as.numeric(test_meta$LOCATION_LAT)
test_lon  <- as.numeric(test_meta$LOCATION_LONG)
test_elev <- as.numeric(test_meta$LOCATION_ELEV)
test_tz   <- TimeZoneCalculatorFLUXNET(test_lat, test_lon)

# Test EFPcalc_year on the first available year
test_yr   <- sort(unique(test_dat$YEAR))[1]
test_dat_yr <- test_dat %>% filter(YEAR == test_yr)

# FIX 4: Correct function call — EFPcalc_year requires all arguments
test_out <- EFPcalc_year(
  dat_year  = test_dat_yr,
  site      = test_site_id,
  yr        = test_yr,
  lat       = test_lat,
  lon       = test_lon,
  elevation = test_elev,
  timezone  = test_tz
)

cat("\nTest output for", test_site_id, test_yr, ":\n")
print(t(test_out))

Test site: AR-TF1 


Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `GPP = clean_col(cur_data(), "GPP_NT_VUT_REF")`.
Caused by warning:
! `cur_data()` was deprecated in dplyr 1.1.0.
ℹ Please use `pick()` instead.”


Rows loaded: 35040 | Years: 2017, 2018 


Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ ins

Quality control:
TA: 179 data points (1.02%) set to NA
H: 207 data points (1.18%) set to NA
LE: 210 data points (1.2%) set to NA
NEE: 301 data points (1.72%) set to NA
-------------------------------------------------------------------
Data filtering:
8448 data points (48.22%) excluded by growing season filter
7856 additional data points (44.84%) excluded by precipitation filter (14682
 data points = 83.8 % in total)
16304 data points (93.06%) excluded in total
1216 valid data points (6.94%) remaining.
Note new column 'z0h' in the function output for 'bigleaf' versions > 0.7.5.
Ground heat flux G is not provided and set to 0.
Energy storage fluxes S are not provided and set to 0.
Respiration from the leaves is ignored and set to 0.
Quality control:
TA: 179 data points (1.02%) set to NA
H: 207 data points (1.18%) set to NA
LE: 210 data points (1.2%) set to NA
NEE: 301 data points (1.72%) set to NA
-------------------------------------------------------------------
Data filtering:
8448 d

New sEddyProc class for site 'AR-TF1'

Start flux partitioning for variable NEE with temperature TA.

Estimate of the temperature sensitivity E_0 from short term data: 97.99.

Regression of reference temperature R_ref for 4 periods.




Test output for AR-TF1 2017 :
            [,1]         
SITE_ID     "AR-TF1"     
YEAR        "2017"       
uWUE        "1.372171"   
WUE         "1.818027"   
ETmax       "0.1828751"  
precipAvail "yes"        
Gavail      "no"         
GSmax       "0.004534142"
CO2avail    "yes"        
G1          "2.148585"   
EF          "0.5637964"  
EFampl      "0.2728276"  
GPPsat      "11.10238"   
NEPmax      "8.869746"   
Rb          "1.149991"   
Rbmax       "1.262253"   
aCUE        "0.5997774"  
TZ          "-3"         
nyears      "1"          
status      "ok"         


## 9. Batch processing — all sites, all years

In [10]:
# ============================================================
# 9. Batch processing
# Writes per-site CSV files immediately so partial runs are not lost.
# Set skip_existing = TRUE to resume an interrupted run.
# ============================================================

skip_existing  <- TRUE
failed_records <- list()

# Process only sites present in BOTH zip_index and tower_meta
sites_to_run <- zip_index %>%
  filter(site_id %in% tower_meta$SITE_ID) %>%
  distinct(site_id, .keep_all = TRUE)

cat("Sites to process:", nrow(sites_to_run), "\n\n")

for (i in seq_len(nrow(sites_to_run))) {

  site_i     <- sites_to_run$site_id[i]
  efp_file_i <- file.path(efp_dir,   paste0(site_i, "_yearly_EFP.csv"))
  met_file_i <- file.path(meteo_dir, paste0(site_i, "_monthly_meteo.csv"))

  if (skip_existing && file.exists(efp_file_i) && file.exists(met_file_i)) {
    message("Skipping (output exists): ", site_i)
    next
  }

  message(sprintf("[%d/%d] Processing: %s", i, nrow(sites_to_run), site_i))

  tryCatch({
    dat_i <- read_fluxmet_hh(
      zip_path   = sites_to_run$zip_path[i],
      hh_file    = sites_to_run$hh_file[i],
      site_id    = site_i,
      start_year = start_year,
      end_year   = end_year
    )

    if (nrow(dat_i) < 48 * 30 * 3)
      stop("Too little data after year filtering: ", nrow(dat_i), " rows")

    efp_i  <- calc_yearly_efps_full(dat_i, site_i, tower_meta)
    meteo_i <- calc_monthly_meteo(dat_i)

    readr::write_csv(efp_i,   efp_file_i)
    readr::write_csv(meteo_i, met_file_i)

  }, error = function(e) {
    msg <- e$message
    message("  FAILED: ", msg)
    failed_records[[length(failed_records) + 1]] <<- tibble(
      site_id  = site_i,
      zip_path = sites_to_run$zip_path[i],
      hh_file  = sites_to_run$hh_file[i],
      error    = msg
    )
  })
}

failed_df <- bind_rows(failed_records)
readr::write_csv(failed_df, file.path(qc_dir, "failed_sites_batch.csv"))

cat("\nBatch complete.\n")
cat("Failed sites this run:", nrow(failed_df), "\n")
if (nrow(failed_df) > 0) print(failed_df)

Sites to process: 329 



[1/329] Processing: AR-TF1

Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It comes from another PROJ installation.”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: /home/nk1125/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 4 is expected. It

  Site: AR-TF1 | Years: 2017, 2018 
Quality control:
TA: 179 data points (1.02%) set to NA
H: 207 data points (1.18%) set to NA
LE: 210 data points (1.2%) set to NA
NEE: 301 data points (1.72%) set to NA
-------------------------------------------------------------------
Data filtering:
8448 data points (48.22%) excluded by growing season filter
7856 additional data points (44.84%) excluded by precipitation filter (14682
 data points = 83.8 % in total)
16304 data points (93.06%) excluded in total
1216 valid data points (6.94%) remaining.
Note new column 'z0h' in the function output for 'bigleaf' versions > 0.7.5.
Ground heat flux G is not provided and set to 0.
Energy storage fluxes S are not provided and set to 0.
Respiration from the leaves is ignored and set to 0.
Quality control:
TA: 179 data points (1.02%) set to NA
H: 207 data points (1.18%) set to NA
LE: 210 data points (1.2%) set to NA
NEE: 301 data points (1.72%) set to NA
------------------------------------------------------

New sEddyProc class for site 'AR-TF1'

Start flux partitioning for variable NEE with temperature TA.

Estimate of the temperature sensitivity E_0 from short term data: 97.99.

Regression of reference temperature R_ref for 4 periods.



Quality control:
TA: 10942 data points (62.45%) set to NA
H: 11322 data points (64.62%) set to NA
LE: 11328 data points (64.66%) set to NA
NEE: 11444 data points (65.32%) set to NA


...

-------------------------------------------------------------------
Data filtering:
12192 data points (69.4%) excluded by growing season filter
1028 additional data points (5.85%) excluded by precipitation filter (2872
 data points = 16.35 % in total)
13220 data points (75.25%) excluded in total
4348 valid data points (24.75%) remaining.
Note new column 'z0h' in the function output for 'bigleaf' versions > 0.7.5.
Energy storage fluxes S are not provided and set to 0.
Respiration from the leaves is ignored and set to 0.
Quality control:
TA: 4748 data points (27.03%) set to NA
H: 5113 data points (29.1%) set to NA
LE: 5202 data points (29.61%) set to NA
NEE: 5622 data points (32%) set to NA


Warning message in filter.growing.season(GPP_daily, tGPP = tGPP, ws = ws, min.int = min.int):
“Attention, there is a gap in 'GPPd' of length n = 79”


-------------------------------------------------------------------
Data filtering:
12192 data points (69.4%) excluded by growing season filter
0 additional data points (0%) excluded by precipitation filter (0 data points = 
0 % in total)
12192 data points (69.4%) excluded in total
5376 valid data points (30.6%) remaining.


New sEddyProc class for site 'ZA-BfS'

Start flux partitioning for variable NEE with temperature TA.

Warning message in fRegrE0fromShortTerm(NightFlux, Temp, DayCounter, ..., MinE0 = MinE0, :
“sMRFluxPartition:::sRegrE0fromShortTerm:::fRegrE0fromShortTerm::: Less than 3 valid values for E_0 after regressing 12 periods! Aborting partitioning.
You may try relaxing the temperature range constraint by setting TempRange = 3 (instead of default 5C). See argument parsE0Regression in sMRFluxPartitioning.”
Estimate of the temperature sensitivity E_0 from short term data: -111.

  REddyProc error for ZA-BfS-2024: In argument: `mean_Rb = mean(Rb_hh, na.rm = TRUE)`.




Batch complete.
Failed sites this run: 0 


## 10. Combine per-site outputs into master tables

In [11]:
# ============================================================
# 10. Combine saved outputs
# ============================================================

efp_files   <- list.files(efp_dir,   pattern = "_yearly_EFP\\.csv$",   full.names = TRUE)
meteo_files <- list.files(meteo_dir, pattern = "_monthly_meteo\\.csv$", full.names = TRUE)

all_efp_df   <- purrr::map_dfr(efp_files,   readr::read_csv, show_col_types = FALSE)
all_meteo_df <- purrr::map_dfr(meteo_files, readr::read_csv, show_col_types = FALSE)

readr::write_csv(all_efp_df,   file.path(out_dir, "ALL_SITES_YEARLY_EFP_corrected.csv"))
readr::write_csv(all_meteo_df, file.path(out_dir, "ALL_SITES_MONTHLY_METEO_corrected.csv"))

cat("Site-years in EFP table:    ", nrow(all_efp_df),   "\n")
cat("Monthly rows in meteo table:", nrow(all_meteo_df), "\n")
dplyr::glimpse(all_efp_df)

Site-years in EFP table:     1722 
Monthly rows in meteo table: 20664 
Rows: 1,722
Columns: 20
$ SITE_ID     <chr> "AR-TF1", "AR-TF1", "AR-TF2", "AR-TF2", "AT-Mmg", "AT-Mmg"…
$ YEAR        <dbl> 2017, 2018, 2017, 2018, 2021, 2022, 2023, 2024, 2018, 2019…
$ uWUE        <dbl> 1.3721711, 2.2942932, 0.6548226, 0.8353541, 2.4393007, 2.9…
$ WUE         <dbl> 1.8180265, 2.3813463, 0.7764370, 0.8397554, 2.8740877, 2.8…
$ ETmax       <dbl> 0.1828751, 0.1648241, 0.1664797, 0.1603725, 0.1989795, 0.2…
$ precipAvail <chr> "yes", "yes", "yes", "yes", "yes", "yes", "yes", "yes", "y…
$ Gavail      <chr> "no", "no", "no", "no", "no", "yes", "yes", "yes", "yes", …
$ GSmax       <dbl> 0.004534142, 0.005551693, 0.004988268, 0.009313015, 0.0079…
$ CO2avail    <chr> "yes", "yes", "yes", "yes", "yes", "yes", "yes", "yes", "y…
$ G1          <dbl> 2.1485846, 2.7675445, 7.9534008, 14.0036634, 2.3253668, 1.…
$ EF          <dbl> 0.5637964, 0.7910594, 0.4856848, 0.7153037, 0.4216831, 0.4…
$ EFampl      <dbl> 0.272

## 11. Quality checks

FIX 8: Column names now match the actual output of `EFPcalc_year()`.

In [12]:
# ============================================================
# 11. Quality checks
# FIX 8: Column names updated to match EFPcalc_year() output
#         (GPPsat, NEPmax, aCUE — not NEP95, CUEeco90)
# ============================================================

# Count non-NA values per EFP
all_efp_df %>%
  summarise(
    n_sites      = n_distinct(SITE_ID),
    n_site_years = n(),
    n_ok         = sum(status == "ok", na.rm = TRUE),
    n_GPPsat     = sum(is.finite(GPPsat),  na.rm = TRUE),
    n_NEPmax     = sum(is.finite(NEPmax),  na.rm = TRUE),
    n_Rb         = sum(is.finite(Rb),      na.rm = TRUE),
    n_aCUE       = sum(is.finite(aCUE),    na.rm = TRUE),
    n_uWUE       = sum(is.finite(uWUE),    na.rm = TRUE),
    n_EF         = sum(is.finite(EF),      na.rm = TRUE),
    n_ETmax      = sum(is.finite(ETmax),   na.rm = TRUE),
    n_GSmax      = sum(is.finite(GSmax),   na.rm = TRUE)
  )


n_sites,n_site_years,n_ok,n_GPPsat,n_NEPmax,n_Rb,n_aCUE,n_uWUE,n_EF,n_ETmax,n_GSmax
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
329,1722,1670,1357,1502,269,249,1356,1663,1667,1554


In [13]:
# Summary statistics by EFP
all_efp_df %>%
  filter(status == "ok") %>%
  summarise(
    across(
      c(GPPsat, NEPmax, Rb, Rbmax, aCUE, WUE, uWUE, EF, EFampl, ETmax, GSmax),
      list(
        median  = ~ median(.x, na.rm = TRUE),
        n_valid = ~ sum(is.finite(.x))
      ),
      .names = "{.col}_{.fn}"
    )
  ) %>%
  tidyr::pivot_longer(everything(),
                      names_to  = c("variable", "stat"),
                      names_sep = "_(?=[^_]+$)") %>%
  tidyr::pivot_wider(names_from = stat, values_from = value) %>%
  print(n = 30)

# A tibble: 22 × 3
   variable   median valid
   <chr>       <dbl> <dbl>
 1 GPPsat   19.5        NA
 2 GPPsat_n NA        1357
 3 NEPmax   17.7        NA
 4 NEPmax_n NA        1502
 5 Rb        2.94       NA
 6 Rb_n     NA         269
 7 Rbmax     4.23       NA
 8 Rbmax_n  NA         269
 9 aCUE      0.379      NA
10 aCUE_n   NA         249
11 WUE       2.65       NA
12 WUE_n    NA        1356
13 uWUE      2.92       NA
14 uWUE_n   NA        1356
15 EF        0.478      NA
16 EF_n     NA        1663
17 EFampl    0.204      NA
18 EFampl_n NA        1663
19 ETmax     0.194      NA
20 ETmax_n  NA        1667
21 GSmax     0.00806    NA
22 GSmax_n  NA        1554
